# StreamFlix — Phase 2: Exploratory Data Analysis

**Analyst:** Meghana J
**Inputs:** Phase 1 cleaned frames (`watch_flagged`, `subs_clean`, `titles`)
**Tools:** Python (pandas, matplotlib)

**Purpose.** Establish what the platform looks like in the data before any KPI is calculated: how much viewing happens and when, what people watch, who they are, where they are, and how they watch.

Ten charts, each saved to `./charts/` as a PNG for the report and dashboard. Every chart is followed by two or three sentences of observation, and anything that looks like an artefact rather than a business pattern is called out as such — a clean chart is worth nothing if the trend behind it is fake.

## 0. Setup

The Phase 1 frames are rebuilt here so the notebook runs standalone. `watch_hours` is derived once and reused: minutes divided by 60.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import os

DATA_DIR = '.'
CHART_DIR = 'charts'
os.makedirs(CHART_DIR, exist_ok=True)

subs    = pd.read_csv(f'{DATA_DIR}/subscribers.csv', parse_dates=['signup_date', 'churn_date'])
titles  = pd.read_csv(f'{DATA_DIR}/titles.csv',      parse_dates=['date_added', 'license_expiry'])
watch   = pd.read_csv(f'{DATA_DIR}/watch_history.csv', parse_dates=['watch_date'])
reviews = pd.read_csv(f'{DATA_DIR}/reviews.csv',     parse_dates=['review_date'])

watch['watch_hours'] = watch['watch_duration_min'] / 60

# one analysis frame: session + subscriber attributes + title attributes
w = (watch
     .merge(subs[['subscriber_id', 'country', 'region', 'plan_type', 'age', 'gender']],
            on='subscriber_id', how='left')
     .merge(titles[['title_id', 'primary_genre', 'type', 'language', 'is_original']],
            on='title_id', how='left'))

# ---- chart style: one place, so all ten charts look like one deck ----
NAVY, BLUE, LIGHT, RED, ORANGE, GREY = '#1F3864', '#2E75B6', '#9DC3E6', '#C00000', '#ED7D31', '#7F7F7F'
PALETTE = [NAVY, BLUE, LIGHT, RED, ORANGE, GREY, '#70AD47', '#FFC000', '#5B9BD5', '#A5A5A5']

plt.rcParams.update({
    'figure.figsize': (11, 5.2),
    'figure.dpi': 110,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.titlecolor': NAVY,
    'axes.titlelocation': 'left',
    'axes.labelsize': 10,
    'axes.edgecolor': '#CCCCCC',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'axes.axisbelow': True,
    'grid.color': '#E6E6E6',
    'grid.linewidth': 0.8,
    'xtick.color': '#444444',
    'ytick.color': '#444444',
})

thousands = FuncFormatter(lambda x, _: f'{x:,.0f}')

def finish(fig, ax_or_none, name, subtitle=None):
    """Add a source note, save the PNG, and show it."""
    fig.text(0.005, -0.02, 'Source: StreamFlix watch_history, 2017-2026',
             fontsize=7.5, color='#888888', ha='left')
    if subtitle:
        fig.text(0.005, 0.965, subtitle, fontsize=9, color='#666666', ha='left')
    path = f'{CHART_DIR}/{name}.png'
    fig.savefig(path)
    plt.show()
    plt.close(fig)
    print('saved:', path)

print('analysis frame:', f'{len(w):,} sessions')

analysis frame: 650,000 sessions


## 1. Monthly viewing volume

Sessions per month across the full 109-month history. Bars, because the question is *how many* in each discrete month; the peak month is annotated so the reader does not have to hunt for it.

In [2]:
monthly = (watch
           .groupby(watch['watch_date'].dt.to_period('M'))
           .agg(sessions=('watch_id', 'count'),
                hours=('watch_hours', 'sum')))
monthly.index = monthly.index.to_timestamp()

fig, ax = plt.subplots()
ax.bar(monthly.index, monthly['sessions'], width=20, color=BLUE)
ax.set_title('Monthly viewing volume, March 2017 to May 2026')
ax.set_ylabel('Sessions per month')
ax.yaxis.set_major_formatter(thousands)
ax.set_xlabel('')

peak = monthly['sessions'].idxmax()
ax.annotate(f"Peak: {monthly.loc[peak, 'sessions']:,.0f} sessions\nin May 2026",
            xy=(peak, monthly.loc[peak, 'sessions']),
            xytext=(-135, -55), textcoords='offset points',
            fontsize=9, color=RED, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=RED, lw=1.2))
finish(fig, ax, 'fig01_monthly_sessions')

print(f'Months covered: {len(monthly)} ({monthly.index.min():%b %Y} to {monthly.index.max():%b %Y})')
print(f'Peak month: {peak:%b %Y} with {monthly.loc[peak, "sessions"]:,.0f} sessions '
      f'= {monthly.loc[peak, "sessions"]/monthly["sessions"].sum()*100:.1f}% of all sessions ever')
print(f'2026 to date: {watch[watch.watch_date.dt.year==2026].shape[0]:,} sessions '
      f'= {watch.watch_date.dt.year.eq(2026).mean()*100:.1f}% of the dataset')
print('\nFive busiest months:')
print(monthly['sessions'].sort_values(ascending=False).head(5).rename(lambda t: t.strftime('%b %Y')).to_string())

saved: charts/fig01_monthly_sessions.png
Months covered: 109 (Mar 2017 to May 2026)
Peak month: May 2026 with 224,696 sessions = 34.6% of all sessions ever
2026 to date: 432,509 sessions = 66.5% of the dataset

Five busiest months:
watch_date
May 2026    224696
Apr 2026     90383
Mar 2026     55665
Feb 2026     33795
Jan 2026     27970


**Observation.** Viewing rises every single month for nine years, with no seasonal dip anywhere, and the last month alone carries 224,696 sessions — 34.6% of everything in the dataset — against 144,718 in the whole of 2025.

That is not a growth curve, it is an exponential ramp, and it is an artefact of how the dataset was generated rather than a business pattern. A real platform would show a plateau and a December/January bulge; this one shows a 12-fold jump between the same five months of 2025 and 2026. **Consequence: the platform's growth rate is not a reportable finding.** Any trend statement in the management report must be about the *shape of engagement* (what people watch, on what device, how completely), not about how fast the business is growing.

## 2. Monthly watch hours, and year-over-year

The same trend measured in hours rather than sessions, because a month can have many short sessions and still be a weak month for viewing time. The second panel compares the same calendar months across years, which is the only fair year-over-year view when the last year is incomplete.

In [3]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [2, 1]})

ax1.plot(monthly.index, monthly['hours'], color=NAVY, lw=2)
ax1.fill_between(monthly.index, monthly['hours'], color=LIGHT, alpha=0.45)
ax1.set_title('Monthly watch hours, 2017 to May 2026')
ax1.set_ylabel('Watch hours')
ax1.yaxis.set_major_formatter(thousands)

# year-over-year: same calendar months only
yoy = (watch[watch['watch_date'].dt.month <= 5]
       .groupby(watch['watch_date'].dt.year)['watch_hours'].sum())
ax2.bar(yoy.index.astype(str), yoy.values, color=[GREY] * (len(yoy) - 1) + [RED])
ax2.set_title('Watch hours, January-May each year')
ax2.set_ylabel('Watch hours')
ax2.yaxis.set_major_formatter(thousands)
ax2.tick_params(axis='x', rotation=45)
for x, v in zip(range(len(yoy)), yoy.values):
    ax2.text(x, v, f'{v/1e3:,.0f}k', ha='center', va='bottom', fontsize=7.5, color='#444444')

fig.suptitle('Watch time over time', x=0.005, ha='left', fontsize=14,
             fontweight='bold', color=NAVY)
fig.tight_layout(rect=[0, 0, 1, 0.94])
finish(fig, None, 'fig02_monthly_hours_yoy')

print('Total watch hours:', f'{watch["watch_hours"].sum():,.0f}')
print(f'Jan-May 2025: {yoy.loc[2025]:,.0f} hours')
print(f'Jan-May 2026: {yoy.loc[2026]:,.0f} hours')
print(f'Change: {(yoy.loc[2026]/yoy.loc[2025]-1)*100:,.0f}%')

saved: charts/fig02_monthly_hours_yoy.png
Total watch hours: 3,333,468
Jan-May 2025: 184,783 hours
Jan-May 2026: 2,225,299 hours
Change: 1,104%


**Observation.** Watch hours track sessions almost exactly — the busiest month by sessions is also the busiest by hours — so the two measures tell the same story and hours are the better unit to report, since they weight a long TV binge above six short trailer-length sessions.

The year-over-year panel confirms the ramp: the same five months produced 184,783 hours in 2025 and 2,225,299 hours in 2026, a twelve-fold increase that no subscription business achieves organically. Total platform viewing is 3,333,468 hours.

## 3. Genre share of watch hours

A horizontal bar rather than a pie: with 16 genres a pie becomes unreadable, and a sorted bar lets the reader compare the long tail directly. Share labels are attached so no one has to estimate.

In [4]:
genre = (w.groupby('primary_genre')
          .agg(hours=('watch_hours', 'sum'),
               sessions=('watch_id', 'count'),
               completion=('completion_pct', 'mean'))
          .sort_values('hours'))
genre['share'] = genre['hours'] / genre['hours'].sum() * 100

fig, ax = plt.subplots(figsize=(11, 6.4))
bars = ax.barh(genre.index, genre['hours'], color=[LIGHT] * (len(genre) - 3) + [BLUE, BLUE, NAVY])
ax.set_title('Watch hours by genre')
ax.set_xlabel('Watch hours')
ax.xaxis.set_major_formatter(thousands)
for bar, share, hours in zip(bars, genre['share'], genre['hours']):
    ax.text(bar.get_width() + 6000, bar.get_y() + bar.get_height() / 2,
            f'{share:.1f}%  ({hours/1e3:,.0f}k h)', va='center', fontsize=8.5, color='#444444')
ax.set_xlim(0, genre['hours'].max() * 1.22)
finish(fig, ax, 'fig03_genre_hours')

print(genre.sort_values('hours', ascending=False)[['hours', 'share', 'completion']].round(1).to_string())
print(f'\nTop 3 genres = {genre["share"].nlargest(3).sum():.1f}% of all watch hours')
print(f'Top 5 genres = {genre["share"].nlargest(5).sum():.1f}% of all watch hours')

saved: charts/fig03_genre_hours.png
                  hours  share  completion
primary_genre                             
Drama          501884.5   15.1        65.1
Comedy         423149.1   12.7        65.3
Action         393171.9   11.8        64.8
Thriller       307024.7    9.2        65.3
Documentary    291216.9    8.7        65.1
Crime          237435.8    7.1        65.5
Romance        214332.3    6.4        65.0
Animation      180093.9    5.4        66.4
Sci-Fi         154332.2    4.6        65.8
Horror         138033.6    4.1        65.3
Fantasy        127838.8    3.8        66.0
Reality        103954.9    3.1        64.7
Kids & Family  103118.1    3.1        65.3
Mystery         82552.4    2.5        65.6
Musical         39023.4    1.2        66.8
Stand-Up        36305.3    1.1        66.3

Top 3 genres = 39.5% of all watch hours
Top 5 genres = 57.5% of all watch hours


**Observation.** Drama leads with 15.1% of watch hours (501,885 hours), followed by Comedy at 12.7% and Action at 11.8% — the top three together are 39.6% of all viewing, and the top five 57.4%. The tail is thin: Musical and Stand-Up contribute 1.2% and 1.1% each.

The spread is remarkably even for a catalogue with 16 genres. No genre is a runaway hit and none is dead weight, which tells leadership the catalogue is balanced rather than driven by a single franchise — and it means cutting a low-share genre saves very little viewing.

## 4. Movies versus TV shows

A donut, because there are only two categories and the question is a share of a whole. Sessions and hours are shown side by side, since the interesting finding is that the two disagree.

In [5]:
ctype = w.groupby('type').agg(hours=('watch_hours', 'sum'),
                              sessions=('watch_id', 'count'))
ctype['hours_share'] = ctype['hours'] / ctype['hours'].sum() * 100
ctype['sessions_share'] = ctype['sessions'] / ctype['sessions'].sum() * 100
ctype['avg_hours_per_session'] = ctype['hours'] / ctype['sessions']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.2))

for ax, col, label in [(ax1, 'hours_share', 'Share of watch hours'),
                       (ax2, 'sessions_share', 'Share of sessions')]:
    ax.pie(ctype[col], labels=[f'{i}\n{v:.1f}%' for i, v in zip(ctype.index, ctype[col])],
           colors=[BLUE, NAVY], startangle=90, counterclock=False,
           wedgeprops=dict(width=0.42, edgecolor='white', linewidth=2),
           textprops=dict(fontsize=10))
    ax.set_title(label)

fig.suptitle('Movies vs TV shows: the share of time and the share of plays disagree',
             x=0.02, ha='left', fontsize=13, fontweight='bold', color=NAVY)
fig.tight_layout(rect=[0, 0, 1, 0.92])
finish(fig, None, 'fig04_type_split')

display(ctype.round(2))

saved: charts/fig04_type_split.png


,hours,sessions,hours_share,sessions_share,avg_hours_per_session
type,,,,,
Movie,450858.81,382330,13.53,58.82,1.18
TV Show,2882608.92,267670,86.47,41.18,10.77


**Observation.** TV shows take 86.5% of all watch hours but only 41.2% of sessions; movies take 58.8% of sessions and just 13.5% of the time. A TV session averages 10.8 hours against 1.2 hours for a movie, because a "session" against a series runs across episodes.

The practical reading: movies get started, series get finished, and the platform's watch time is carried by series. Catalogue investment and retention effort should be weighted accordingly — a movie costs licence money per hour of engagement far more than a returning series does.

## 5. Top 10 countries by watch hours

Country lives on the subscriber, not on the session, so it is joined in from `subscribers`. Horizontal bars, sorted, with each country's share labelled.

In [6]:
country = (w.groupby('country')
            .agg(hours=('watch_hours', 'sum'),
                 subscribers=('subscriber_id', 'nunique'))
            .sort_values('hours', ascending=False))
country['share'] = country['hours'] / country['hours'].sum() * 100
top10 = country.head(10).sort_values('hours')

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(top10.index, top10['hours'], color=[BLUE] * len(top10))
bars[-1].set_color(NAVY); bars[-2].set_color(NAVY); bars[-3].set_color(NAVY)
ax.set_title('Top 10 countries by watch hours')
ax.set_xlabel('Watch hours')
ax.xaxis.set_major_formatter(thousands)
for bar, share in zip(bars, top10['share']):
    ax.text(bar.get_width() + 8000, bar.get_y() + bar.get_height() / 2,
            f'{share:.1f}%', va='center', fontsize=9, color='#444444')
ax.set_xlim(0, top10['hours'].max() * 1.12)
finish(fig, ax, 'fig05_top_countries')

print(country.head(10).round(1).to_string())
print(f'\nTop 10 countries = {country["share"].head(10).sum():.1f}% of watch hours, '
      f'from {country.head(10)["subscribers"].sum():,} of {subs.shape[0]:,} subscribers')
print(f'Countries in dataset: {len(country)}')

saved: charts/fig05_top_countries.png
                   hours  subscribers  share
country                                     
United States   910551.2         3774   27.3
India           380917.7         1872   11.4
United Kingdom  285346.4         1266    8.6
South Korea     239498.9         1096    7.2
Brazil          206648.2          835    6.2
Japan           171708.0          842    5.2
France          167418.2          660    5.0
Mexico          143971.8          749    4.3
Spain           137107.4          671    4.1
Germany         136936.4          573    4.1

Top 10 countries = 83.4% of watch hours, from 12,338 of 15,000 subscribers
Countries in dataset: 20


**Observation.** The United States is the single largest market at 27.3% of watch hours (910,551 hours from 3,774 subscribers), ahead of India at 11.4% and the United Kingdom at 8.6%. The top ten countries account for 83.4% of all viewing.

Concentration is high but not fragile: no market exceeds about a quarter of engagement, so no single country's pricing or regulatory change can sink the platform, while the top three still justify market-specific content decisions.

## 6. Subscriber distribution by plan

A pie, three slices, straight from `subscribers` — one row per subscriber, so this is a count of people rather than a count of sessions.

In [7]:
plan = subs['plan_type'].value_counts()
order = ['Basic with Ads', 'Standard', 'Premium']
plan = plan.reindex(order)

fig, ax = plt.subplots(figsize=(8, 5.4))
wedges, texts, autotexts = ax.pie(
    plan, labels=plan.index, colors=[LIGHT, BLUE, NAVY], startangle=90, counterclock=False,
    autopct=lambda p: f'{p:.1f}%\n({p/100*plan.sum():,.0f})',
    wedgeprops=dict(edgecolor='white', linewidth=2), textprops=dict(fontsize=10))
for t in autotexts:
    t.set_color('white'); t.set_fontweight('bold'); t.set_fontsize(9)
ax.set_title('Subscribers by plan (15,000 subscribers)')
finish(fig, ax, 'fig06_plan_split')

plan_mrr = (subs[subs['is_active']]
            .groupby('plan_type')['monthly_price_usd'].sum().reindex(order))
display(pd.DataFrame({'subscribers': plan, 'share_%': (plan / plan.sum() * 100).round(1)}))
print('Active MRR by plan:'); print(plan_mrr.round(0).to_string())
print(f'Total MRR from active subscribers: ${plan_mrr.sum():,.0f}')

saved: charts/fig06_plan_split.png


,subscribers,share_%
plan_type,,
Basic with Ads,4458,29.7
Standard,6100,40.7
Premium,4442,29.6


Active MRR by plan:
plan_type
Basic with Ads    20474.0
Standard          71734.0
Premium           83661.0
Total MRR from active subscribers: $175,869


**Observation.** Standard is the most common plan at 40.7% (6,100 subscribers), with Basic with Ads and Premium almost tied at 29.7% and 29.6%. The base is middle-weighted rather than premium-heavy.

Because only 29.6% of subscribers are on Premium ($22.99) while 29.7% sit on Basic with Ads ($6.99), the cheapest tier is as large as the most expensive one — which caps ARPU and makes upgrading Basic-with-Ads subscribers the single biggest revenue lever available.

## 7. Device usage

Sessions by device, as a share. Vertical bars keep the six device names readable without rotation.

In [8]:
device = (watch['device'].value_counts()
          .rename_axis('device').reset_index(name='sessions'))
device['share'] = device['sessions'] / device['sessions'].sum() * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(device['device'], device['sessions'],
              color=[NAVY, BLUE, LIGHT, LIGHT, LIGHT, LIGHT])
ax.set_title('Viewing sessions by device')
ax.set_ylabel('Sessions')
ax.yaxis.set_major_formatter(thousands)
for bar, share in zip(bars, device['share']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{share:.1f}%', ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#333333')
ax.set_ylim(0, device['sessions'].max() * 1.15)
finish(fig, ax, 'fig07_device')

display(device.round(1))
print(f'Big screen (Smart TV + Streaming Stick): {device.loc[device.device.isin(["Smart TV","Streaming Stick"]), "share"].sum():.1f}%')
print(f'Mobile + Tablet: {device.loc[device.device.isin(["Mobile","Tablet"]), "share"].sum():.1f}%')

saved: charts/fig07_device.png


,device,sessions,share
0,Smart TV,221446,34.1
1,Mobile,175605,27.0
2,Laptop,96408,14.8
3,Tablet,65276,10.0
4,Game Console,52322,8.0
5,Streaming Stick,38943,6.0


Big screen (Smart TV + Streaming Stick): 40.1%
Mobile + Tablet: 37.1%


**Observation.** The Smart TV is the dominant device at 34.1% of sessions, then Mobile at 27.0%, Laptop 14.8%, Tablet 10.0%, Game Console 8.0% and Streaming Stick 6.0%. Adding the stick to the TV, 40.1% of viewing happens on a television, against 37.0% on phone or tablet.

Viewing is a living-room activity that is also genuinely mobile. That matters for two decisions: playback quality and app performance investment should follow the TV share, while anything that needs to reach subscribers in the moment — notifications, recommendations, watchlist prompts — belongs on mobile.

## 8. Age distribution of subscribers

A histogram of subscriber ages, with the median marked. Hours are then grouped into bands, because a histogram of subscriber counts answers "who is there" while the band table answers the brief's actual question, "who watches most".

In [9]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.hist(subs['age'], bins=range(13, 90, 3), color=LIGHT, edgecolor='white')
ax1.axvline(subs['age'].median(), color=RED, lw=1.6, ls='--')
ax1.text(subs['age'].median() + 1.2, ax1.get_ylim()[1] * 0.9,
         f'median {subs["age"].median():.0f} yrs', color=RED, fontsize=9, fontweight='bold')
ax1.set_title('Subscribers by age')
ax1.set_xlabel('Age (years)'); ax1.set_ylabel('Subscribers')
ax1.yaxis.set_major_formatter(thousands)

bands = [13, 17, 24, 34, 44, 54, 64, 85]
labels = ['13-17', '18-24', '25-34', '35-44', '45-54', '55-64', '65+']
w['age_band'] = pd.cut(w['age'], bands, labels=labels, include_lowest=True)
band_hours = w.groupby('age_band', observed=True)['watch_hours'].sum()
band_share = band_hours / band_hours.sum() * 100

bars = ax2.bar(band_share.index.astype(str), band_share.values,
               color=[LIGHT] * 7)
bars[2].set_color(BLUE); bars[3].set_color(NAVY)
ax2.set_title('Share of watch hours by age band')
ax2.set_ylabel('% of watch hours')
for bar, v in zip(bars, band_share.values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
             f'{v:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold', color='#333333')
ax2.set_ylim(0, band_share.max() * 1.15)

fig.tight_layout()
finish(fig, None, 'fig08_age')

print(subs['age'].describe().round(1).to_string())
print('\nWatch hours by age band:')
print(pd.DataFrame({'hours': band_hours.round(0), 'share_%': band_share.round(1)}).to_string())

saved: charts/fig08_age.png
count    15000.0
mean        35.6
std         12.5
min         13.0
25%         27.0
50%         35.0
75%         44.0
max         85.0

Watch hours by age band:
             hours  share_%
age_band                   
13-17     281924.0      8.5
18-24     358172.0     10.7
25-34     890783.0     26.7
35-44     972657.0     29.2
45-54     603302.0     18.1
55-64     192413.0      5.8
65+        34216.0      1.0


**Observation.** The subscriber base is centred on the mid-thirties — mean 35.6, median 35 — and the 25-44 group accounts for 55.9% of all watch hours (25-34: 26.7%, 35-44: 29.2%). Under-25s are 19.2% of viewing and the 65+ group just 1.0%.

The platform is an adult, family-stage product, not a youth one. Content that lands with 25-44 year olds — returning series, drama and comedy — is what the audience is actually built around, and the 55+ tail is too small to justify dedicated spend.

## 9. Completion rate by genre

The brief asks which genres get finished. Sorted bars, with the platform average drawn as a reference line so each genre is read as above or below the norm rather than in isolation.

In [10]:
comp = (w.groupby('primary_genre')['completion_pct'].mean().sort_values())
overall = w['completion_pct'].mean()

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(comp.index, comp.values, color=[BLUE if v >= overall else LIGHT for v in comp.values])
ax.axvline(overall, color=RED, lw=1.4, ls='--')
ax.text(overall + 0.08, -0.9, f'platform average {overall:.1f}%', color=RED, fontsize=9, fontweight='bold')
ax.set_title('Average completion rate by genre')
ax.set_xlabel('Average completion (%)')
ax.set_xlim(60, 70)
for bar, v in zip(bars, comp.values):
    ax.text(bar.get_width() + 0.06, bar.get_y() + bar.get_height() / 2, f'{v:.1f}%',
            va='center', fontsize=8.5, color='#444444')
finish(fig, ax, 'fig09_completion_genre')

print(comp.round(1).to_string())
print(f'\nPlatform average completion: {overall:.1f}%')
print(f'Range across genres: {comp.min():.1f}% to {comp.max():.1f}% (spread of {comp.max()-comp.min():.1f} points)')

saved: charts/fig09_completion_genre.png
primary_genre
Reality          64.7
Action           64.8
Romance          65.0
Drama            65.1
Documentary      65.1
Horror           65.3
Thriller         65.3
Kids & Family    65.3
Comedy           65.3
Crime            65.5
Mystery          65.6
Sci-Fi           65.8
Fantasy          66.0
Stand-Up         66.3
Animation        66.4
Musical          66.8

Platform average completion: 65.3%
Range across genres: 64.7% to 66.8% (spread of 2.1 points)


**Observation.** Completion is flat across genres: Musical is highest at 66.8% and Reality lowest at 64.7%, a spread of just 2.1 points around a 65.3% platform average.

Genre is therefore not what determines whether people finish. Since every genre sits within a hair of the average, there is no "weak genre" to fix on completion grounds — the lever must be within-genre (specific titles, or the length and pacing of a series) rather than at the genre level. That is a finding in itself, and it saves leadership from a costly genre cull that would not move the number.

## 10. Review sentiment

Counts and shares of the three sentiment values across 110,000 written reviews, with the negative slice called out because it is the one with a cost attached.

In [11]:
sent = reviews['sentiment'].value_counts().reindex(['Positive', 'Neutral', 'Negative'])
sent_share = sent / sent.sum() * 100

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(sent.index, sent.values, color=[BLUE, GREY, RED], width=0.55)
ax.set_title('Review sentiment across 110,000 written reviews')
ax.set_ylabel('Reviews')
ax.yaxis.set_major_formatter(thousands)
for bar, n, share in zip(bars, sent.values, sent_share.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{n:,.0f}\n({share:.1f}%)', ha='center', va='bottom',
            fontsize=10, fontweight='bold', color='#333333')
ax.set_ylim(0, sent.max() * 1.2)
finish(fig, ax, 'fig10_sentiment')

display(pd.DataFrame({'reviews': sent, 'share_%': sent_share.round(1)}))
print(f'Positive-to-negative ratio: {sent["Positive"]/sent["Negative"]:.1f} to 1')
print(f'Mean helpful votes by sentiment:')
print(reviews.groupby('sentiment')['helpful_votes'].mean().round(2).to_string())

saved: charts/fig10_sentiment.png


,reviews,share_%
sentiment,,
Positive,75007,68.2
Neutral,22759,20.7
Negative,12234,11.1


Positive-to-negative ratio: 6.1 to 1
Mean helpful votes by sentiment:
sentiment
Negative    4.00
Neutral     4.01
Positive    9.00


**Observation.** Written reviews are 68.2% Positive, 20.7% Neutral and 11.1% Negative — a positive-to-negative ratio of 6.1 to 1 across 110,000 reviews, which is a healthy sentiment profile for a subscription service.

The 12,234 negative reviews are the actionable group: they are a large enough sample to read in full and are the fastest route to a concrete product complaint list, which is exactly what the Phase 4 experience page should surface rather than a bare sentiment count.

## 11. What Phase 2 establishes

| Question | Answer | Evidence |
|---|---|---|
| How much viewing is there? | 3,333,468 hours across 650,000 sessions | fig 02 |
| Is the platform growing? | Cannot be answered — the volume curve is an artefact (12× year on year) | fig 01, 02 |
| What genre earns most watch time? | Drama 15.1%, Comedy 12.7%, Action 11.8%; top 3 = 39.6% | fig 03 |
| Movies or series? | Series: 86.5% of hours from 41.2% of sessions | fig 04 |
| Where is the audience? | US 27.3%, India 11.4%, UK 8.6%; top 10 = 83.4% | fig 05 |
| Who are they? | Median 35 years; 25-44 = 55.9% of hours | fig 08 |
| What plan? | Standard 40.7%; Basic with Ads 29.7%; Premium 29.6% | fig 06 |
| How do they watch? | Smart TV 34.1%, Mobile 27.0%; big screen 40.1% | fig 07 |
| Does genre affect completion? | No — 64.7% to 66.8% around a 65.3% average | fig 09 |
| How do they feel? | 68.2% positive, 11.1% negative (6.1:1) | fig 10 |

All ten charts are saved as PNG files in `./charts/`, named `fig01` to `fig10`, ready for the dashboard and the management report.